# DATA ACCESS AND STORAGE PROJECT
## Building Reliable Analytical Data Pipelines for Data Science • Business • Economics • Supply Chain

In practice, a data analyst is rarely asked to perform an isolated task such as:

> “Read this CSV file.”

or:

> “Save this DataFrame to Excel.”

Instead, the analyst usually receives a **decision-oriented request** such as:

- Build a reliable sales dataset for the management team.
- Combine economic indicators published in different formats.
- Prepare a modeling table before a Machine Learning project starts.
- Integrate inventory, supplier, and shipment data for operational monitoring.

The challenge is not only to read files. The analyst must determine:

- where the required data are stored;
- how different sources are structured;
- which fields can be used as keys;
- whether data types are correct;
- whether missing or duplicated values exist;
- how sources should be integrated;
- how the resulting analytical data should be stored for reuse.

This notebook contains **four mini-projects**:

1. **Business Analytics** — Integrating retail customers, products, and orders.
2. **Economics** — Harmonizing macroeconomic indicators from CSV, JSON, and HTML.
3. **Data Science** — Building and validating a modeling dataset from multiple sources.
4. **Supply Chain Analytics** — Creating an operational data mart from inventory, shipment, and supplier data.

> This is a **guided independent project**. Cells marked `TODO` are intentionally incomplete and should be completed by the learner.

## Learning Objectives

After completing the project, you should be able to:

- start from a business or analytical question rather than from a file format;
- identify suitable data sources for an analytical task;
- read CSV, Excel, JSON, HTML, and SQLite data;
- control data types, dates, delimiters, and missing-value markers during import;
- inspect source structure before integration;
- validate primary keys and foreign keys;
- flatten nested JSON structures;
- use SQL and parameterized queries;
- integrate multiple sources with appropriate joins;
- detect unmatched records and data-quality problems;
- build reusable analytical datasets;
- store results in CSV, Excel, and SQLite;
- explain why a storage choice is appropriate for a given use case;
- distinguish **data availability** from **data readiness**.

## General Approach for Each Mini-Project

Each mini-project follows a realistic analytical workflow.

### Step 1 — Understand the Context

Before writing code, identify:

- Who will use the final dataset?
- What decision or analysis will it support?
- Which source contains each required variable?

### Step 2 — Translate the Problem into Data Requirements

For example:

> “Build a reliable sales report.”

must be translated into more specific questions:

- Where are customer attributes stored?
- Where are product prices stored?
- Where are order items stored?
- Which keys link these sources?
- Which variables need to be created after integration?

### Step 3 — Access and Inspect the Sources

Do not merge immediately.

First inspect:

- `head()`;
- `shape`;
- `dtypes`;
- missing values;
- unique keys;
- date formats;
- inconsistent category labels.

### Step 4 — Transform and Integrate

Typical tasks include:

- parsing dates;
- normalizing identifiers;
- flattening nested JSON;
- filtering SQL data;
- renaming columns;
- merging tables;
- validating relationships.

### Step 5 — Validate the Result

A successful `merge()` does not automatically mean the result is correct.

Check:

- row counts;
- unmatched keys;
- missing analytical fields;
- duplicated keys;
- impossible values;
- consistency between sources.

### Step 6 — Store and Document

Choose storage according to purpose:

- **CSV** → simple data exchange;
- **Excel** → multi-sheet business reporting;
- **SQLite/SQL** → reusable relational analytics and querying;
- **JSON** → nested data exchange;
- **Parquet** → efficient analytical storage when supported.

### General Rules

- Use `pandas`, `numpy`, `json`, and `sqlite3` for the core tasks.
- Keep identifiers such as CustomerID or ProductID as strings when appropriate.
- Use parameterized SQL when query values come from variables.
- Do not treat a successful file import as evidence that the data are correct.
- Write conclusions in the form:

**Data issue / Observation → Evidence → Analytical or business implication**

## 0. Environment Setup

In [ ]:
import sys
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

np.random.seed(42)

BASE_DIR = Path("data_access_storage_project")
SOURCE_DIR = BASE_DIR / "sources"
OUTPUT_DIR = BASE_DIR / "outputs"

SOURCE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Project directory:", BASE_DIR.resolve())

## 0.1 Create the Project Data Sources

The notebook creates all required source files so that it can run independently.

The purpose of the following cell is **not** to test your ability to generate synthetic data.

Its role is to create realistic sources with:

- different file formats;
- different date representations;
- missing-value markers;
- nested JSON;
- inconsistent identifiers;
- relational database tables;
- a few deliberate data-quality issues.

Your task begins when you start accessing and validating these sources.

In [ ]:
rng = np.random.default_rng(42)

# ============================================================
# MINI-PROJECT 1 SOURCES — RETAIL
# ============================================================

retail_customers = pd.DataFrame({
    "CustomerID": ["C001", "C002", "C003", "C004", "C005", "C006", "C007", "C008"],
    "CustomerName": [
        "An Nguyen", "Binh Tran", "Chi Le", "Dung Pham",
        "Giang Vu", "Ha Do", "Khanh Hoang", "Linh Bui"
    ],
    "City": ["Hanoi", "Hanoi", "Danang", "HCMC", "HCMC", "Danang", "Hanoi", "HCMC"],
    "Segment": ["Retail", "Corporate", "Retail", "SME", "Corporate", "Retail", "SME", "Corporate"],
    "SignupDate": [
        "2025-01-10", "2025-02-18", "2025-03-07", "2025-03-21",
        "2025-04-15", "2025-05-09", "2025-06-01", "2025-06-19"
    ]
})
retail_customers.to_csv(SOURCE_DIR / "retail_customers.csv", index=False)

retail_products = pd.DataFrame({
    "ProductID": ["P01", "P02", "P03", "P04", "P05"],
    "ProductName": ["Laptop", "Monitor", "Keyboard", "Mouse", "Headset"],
    "Category": ["Computers", "Accessories", "Accessories", "Accessories", "Accessories"],
    "UnitPrice": [1200, 320, 80, 35, 95]
})

retail_inventory = pd.DataFrame({
    "ProductID": ["P01", "P02", "P03", "P04", "P05"],
    "Stock": [10, 24, 70, 92, 38],
    "ReorderLevel": [5, 10, 20, 30, 15]
})

with pd.ExcelWriter(SOURCE_DIR / "retail_products.xlsx", engine="openpyxl") as writer:
    retail_products.to_excel(writer, sheet_name="Products", index=False)
    retail_inventory.to_excel(writer, sheet_name="Inventory", index=False)

retail_orders = [
    {
        "OrderID": "O001",
        "OrderDate": "2025-07-01",
        "Customer": {"CustomerID": "C001", "City": "Hanoi"},
        "Items": [
            {"ProductID": "P01", "Quantity": 1},
            {"ProductID": "P04", "Quantity": 2}
        ]
    },
    {
        "OrderID": "O002",
        "OrderDate": "2025-07-02",
        "Customer": {"CustomerID": "C003", "City": "Danang"},
        "Items": [
            {"ProductID": "P02", "Quantity": 2},
            {"ProductID": "P03", "Quantity": 3}
        ]
    },
    {
        "OrderID": "O003",
        "OrderDate": "2025-07-03",
        "Customer": {"CustomerID": "C008", "City": "HCMC"},
        "Items": [
            {"ProductID": "P05", "Quantity": 2},
            {"ProductID": "P04", "Quantity": 1}
        ]
    },
    {
        "OrderID": "O004",
        "OrderDate": "2025-07-04",
        "Customer": {"CustomerID": "C006", "City": "Danang"},
        "Items": [
            {"ProductID": "P03", "Quantity": 1},
            {"ProductID": "P04", "Quantity": 2}
        ]
    }
]

with open(SOURCE_DIR / "retail_orders.json", "w", encoding="utf-8") as f:
    json.dump(retail_orders, f, ensure_ascii=False, indent=2)

# ============================================================
# MINI-PROJECT 2 SOURCES — ECONOMICS
# ============================================================

quarters = pd.period_range("2022Q1", "2025Q4", freq="Q").astype(str)
t = np.arange(len(quarters))

inflation = 2.5 + 3.5 * np.exp(-((t - 6) / 2.8) ** 2) + rng.normal(0, 0.18, len(t))
unemployment = 5.2 - 0.10 * t + 0.9 * np.exp(-((t - 11) / 2.5) ** 2) + rng.normal(0, 0.12, len(t))
gdp = 4.5 + 0.5 * np.sin(t / 2) - 2.2 * np.exp(-((t - 11) / 2.0) ** 2) + rng.normal(0, 0.22, len(t))

inflation_df = pd.DataFrame({
    "Quarter": quarters,
    "Inflation": np.round(inflation, 2)
})
inflation_df.loc[5, "Inflation"] = np.nan
inflation_df.to_csv(SOURCE_DIR / "inflation.csv", index=False)

unemployment_json = [
    {
        "period": q,
        "labor_market": {
            "unemployment_rate": round(float(u), 2)
        }
    }
    for q, u in zip(quarters, unemployment)
]
with open(SOURCE_DIR / "unemployment.json", "w", encoding="utf-8") as f:
    json.dump(unemployment_json, f, indent=2)

gdp_df = pd.DataFrame({
    "Period": quarters,
    "GDP Growth (%)": np.round(gdp, 2)
})
html = "<html><body><h1>Quarterly GDP Report</h1>" + gdp_df.to_html(index=False) + "</body></html>"
(SOURCE_DIR / "gdp_report.html").write_text(html, encoding="utf-8")

# ============================================================
# MINI-PROJECT 3 SOURCES — DATA SCIENCE MODELING DATA
# ============================================================

n = 600
customer_ids = [f"U{i:04d}" for i in range(1, n + 1)]

ds_customer_master = pd.DataFrame({
    "CustomerID": customer_ids,
    "Age": rng.integers(18, 70, n),
    "City": rng.choice(["Hanoi", "Danang", "HCMC"], n, p=[0.35, 0.20, 0.45]),
    "JoinDate": pd.date_range("2023-01-01", periods=n, freq="D").astype(str),
    "Plan": rng.choice(["Basic", "Plus", "Premium"], n, p=[0.45, 0.35, 0.20])
})
ds_customer_master.loc[20, "Age"] = np.nan
ds_customer_master.loc[40, "City"] = None
ds_customer_master.to_csv(SOURCE_DIR / "ds_customer_master.csv", index=False)

tenure = rng.integers(1, 48, n)
monthly_charge = np.clip(rng.normal(65, 18, n), 20, 120)
support_calls = rng.poisson(1.8, n)

ds_behavior = pd.DataFrame({
    "CustomerID": customer_ids,
    "Tenure": tenure,
    "MonthlyCharge": np.round(monthly_charge, 2),
    "SupportCalls": support_calls,
    "LastLoginDays": rng.integers(0, 60, n)
})
ds_behavior.to_csv(SOURCE_DIR / "ds_behavior.csv", sep=";", index=False)

logit = (
    -1.6
    + 0.018 * (monthly_charge - 60)
    - 0.04 * (tenure - 20)
    + 0.28 * support_calls
)
p_churn = 1 / (1 + np.exp(-logit))
labels = rng.binomial(1, p_churn)

conn_labels = sqlite3.connect(SOURCE_DIR / "ml_labels.db")
pd.DataFrame({
    "CustomerID": customer_ids,
    "Churn": labels
}).to_sql("churn_labels", conn_labels, if_exists="replace", index=False)
conn_labels.close()

# ============================================================
# MINI-PROJECT 4 SOURCES — SUPPLY CHAIN
# ============================================================

inventory_sc = pd.DataFrame({
    "SKU": ["S001", "S002", "S003", "S004", "S005", "S006"],
    "Product": ["Item A", "Item B", "Item C", "Item D", "Item E", "Item F"],
    "Warehouse": ["North", "North", "Central", "Central", "South", "South"],
    "OnHand": [45, 12, 8, 60, 25, 6],
    "ReorderPoint": [20, 15, 12, 25, 18, 10]
})

supplier_sc = pd.DataFrame({
    "SupplierID": ["SUP01", "SUP02", "SUP03"],
    "SupplierName": ["Alpha Supply", "Beta Industrial", "Gamma Trading"],
    "Country": ["Vietnam", "Thailand", "Singapore"]
})

with pd.ExcelWriter(SOURCE_DIR / "supply_chain_master.xlsx", engine="openpyxl") as writer:
    inventory_sc.to_excel(writer, sheet_name="Inventory", index=False)
    supplier_sc.to_excel(writer, sheet_name="Suppliers", index=False)

shipments_sc = [
    {
        "ShipmentID": "SH001",
        "SupplierID": "SUP01",
        "ExpectedDate": "2025-08-01",
        "ActualDate": "2025-08-02",
        "Lines": [
            {"SKU": "S001", "Qty": 20},
            {"SKU": "S002", "Qty": 30}
        ]
    },
    {
        "ShipmentID": "SH002",
        "SupplierID": "SUP02",
        "ExpectedDate": "2025-08-03",
        "ActualDate": "2025-08-09",
        "Lines": [
            {"SKU": "S003", "Qty": 25},
            {"SKU": "S004", "Qty": 10}
        ]
    },
    {
        "ShipmentID": "SH003",
        "SupplierID": "SUP03",
        "ExpectedDate": "2025-08-05",
        "ActualDate": "2025-08-05",
        "Lines": [
            {"SKU": "S005", "Qty": 20},
            {"SKU": "S006", "Qty": 18}
        ]
    }
]
with open(SOURCE_DIR / "shipments.json", "w", encoding="utf-8") as f:
    json.dump(shipments_sc, f, indent=2)

conn_ops = sqlite3.connect(SOURCE_DIR / "operations.db")
pd.DataFrame({
    "Warehouse": ["North", "Central", "South"],
    "Manager": ["Lan", "Minh", "Hoa"],
    "Capacity": [500, 420, 460]
}).to_sql("warehouses", conn_ops, if_exists="replace", index=False)
conn_ops.close()

print("Source files created:")
for path in sorted(SOURCE_DIR.iterdir()):
    print("-", path.name)

# MINI-PROJECT 1 — BUSINESS ANALYTICS
## Build a Reliable Retail Sales Data Mart

### Scenario

A retail company stores customer, product, inventory, and order information in different systems:

- customer master data → CSV;
- product and inventory data → Excel;
- online orders → nested JSON.

Management wants a reusable analytical dataset for monthly reporting.

The current problem is not a lack of data. The problem is that the required information is **distributed across multiple sources**.

### Decision Context

Management wants the data mart to support questions such as:

- Which customers and cities generate the most revenue?
- Which product categories contribute the most sales?
- Are order records linked to valid customer and product master data?
- Can the same analytical table be reused in SQL and Excel reporting?

### Analytical Data Requirements

The final table should contain:

- `OrderID`
- `OrderDate`
- `CustomerID`
- `CustomerName`
- `City`
- `Segment`
- `ProductID`
- `ProductName`
- `Category`
- `Quantity`
- `UnitPrice`
- `Revenue`

### Required Deliverables

1. A validated DataFrame named `retail_analytics`.
2. A SQLite database containing the detailed analytical table.
3. An Excel workbook containing detail and KPI worksheets.
4. A short **3–5 bullet data-quality and business summary**.

> The objective is not merely to make `merge()` run. The objective is to produce a dataset that can be trusted and reused.

## 1.1 Inspect the Customer CSV

Before importing the data, answer:

- Which field is the customer key?
- Should it be numerical or text?
- Which field should be parsed as a date?

In [ ]:
# TODO 1.1
# customers_1 = pd.read_csv(
#     SOURCE_DIR / "retail_customers.csv",
#     dtype={...},
#     parse_dates=[...]
# )
#
# print(customers_1.head())
# print(customers_1.shape)
# print(customers_1.dtypes)
# print(customers_1.isna().sum())

### Questions 1.1

- Is `CustomerID` unique?
- Are any customer records missing important fields?
- Why should an identifier usually not be treated as a numerical measurement?

**Your answer:**

> ...

## 1.2 Inspect the Excel Workbook

The workbook contains two worksheets:

- `Products`
- `Inventory`

### Tasks

1. Read the entire workbook using `sheet_name=None`.
2. Print worksheet names.
3. Check the shape of each worksheet.
4. Validate that `ProductID` is unique in the product master.

In [ ]:
# TODO 1.2
# retail_book = pd.read_excel(
#     SOURCE_DIR / "retail_products.xlsx",
#     sheet_name=None,
#     engine="openpyxl"
# )
#
# print(...)
#
# products_1 = retail_book["Products"]
# inventory_1 = retail_book["Inventory"]
#
# print("Product key unique:", ...)

## 1.3 Flatten the Nested Order JSON

Each order contains a nested `Items` list.

The analytical table requires **one row per order item**.

### Tasks

1. Read `retail_orders.json`.
2. Use `pd.json_normalize()`.
3. Retain parent-level fields:
   - OrderID;
   - OrderDate;
   - CustomerID;
   - City.
4. Convert `OrderDate` to datetime.

In [ ]:
# TODO 1.3
# with open(SOURCE_DIR / "retail_orders.json", encoding="utf-8") as f:
#     orders_1 = json.load(f)
#
# order_items_1 = pd.json_normalize(
#     orders_1,
#     record_path=[...],
#     meta=[...]
# )
#
# order_items_1 = order_items_1.rename(columns={...})
# order_items_1["OrderDate"] = pd.to_datetime(...)
#
# print(order_items_1)

### Questions 1.3

- How many orders exist?
- How many order-item rows exist after normalization?
- Why are these two counts different?

> ...

## 1.4 Validate Keys Before Merging

A merge should not begin until key relationships are understood.

### Tasks

Check:

1. `CustomerID` is unique in the customer master.
2. `ProductID` is unique in the product master.
3. Every order customer exists in the customer master.
4. Every order product exists in the product master.

In [ ]:
# TODO 1.4
# customer_ids_orders = set(order_items_1["CustomerID"])
# customer_ids_master = set(customers_1["CustomerID"])
#
# product_ids_orders = set(order_items_1["ProductID"])
# product_ids_master = set(products_1["ProductID"])
#
# missing_customers = ...
# missing_products = ...
#
# print("Missing customers:", missing_customers)
# print("Missing products:", missing_products)

## 1.5 Build the Retail Analytical Dataset

### Tasks

1. Merge order items with products.
2. Merge the result with customers.
3. Use `validate="many_to_one"` where appropriate.
4. Create:

\[
Revenue = Quantity \times UnitPrice
\]

5. Store the result as `retail_analytics`.

In [ ]:
# TODO 1.5
# retail_analytics = (
#     order_items_1
#     .merge(
#         products_1,
#         on="ProductID",
#         how="left",
#         validate="many_to_one"
#     )
#     .merge(
#         customers_1[...],
#         on="CustomerID",
#         how="left",
#         validate="many_to_one"
#     )
# )
#
# retail_analytics["Revenue"] = ...
#
# print(retail_analytics.head())
# print(retail_analytics.isna().sum())

## 1.6 Create KPI Tables and Store the Results

Create:

### City KPI

- Revenue;
- unique orders;
- unique customers.

### Category KPI

- Revenue;
- total quantity.

### Storage Requirements

Save:

- detailed table → SQLite table `retail_analytics`;
- city KPI → SQLite table `city_kpi`;
- category KPI → SQLite table `category_kpi`;
- all three tables → separate worksheets in `retail_dashboard.xlsx`.

In [ ]:
# TODO 1.6
# city_kpi_1 = ...
# category_kpi_1 = ...
#
# conn = sqlite3.connect(OUTPUT_DIR / "retail_analytics.db")
# retail_analytics.to_sql(...)
# city_kpi_1.to_sql(...)
# category_kpi_1.to_sql(...)
# conn.close()
#
# with pd.ExcelWriter(
#     OUTPUT_DIR / "retail_dashboard.xlsx",
#     engine="openpyxl"
# ) as writer:
#     ...

### Mini-Project 1 Conclusion

Write **3–5 bullets** following:

- **Observation:** what did you find?
- **Evidence:** which validation result or KPI supports it?
- **Implication:** why does it matter for reporting or management?

1. ...
2. ...
3. ...

# MINI-PROJECT 2 — ECONOMICS
## Harmonize Economic Indicators Published in Different Formats

### Scenario

An economics research team receives three quarterly indicators from different publication systems:

- Inflation → CSV;
- Unemployment → nested JSON;
- GDP Growth → HTML table.

The team needs one consistent quarterly dataset before producing an economic briefing.

### Main Data Problem

The issue is not simply reading three files.

The sources differ in:

- format;
- field names;
- nesting;
- missing values;
- column naming conventions.

### Analytical Questions

1. Are the same quarters available in every source?
2. Are any observations missing?
3. Can the three indicators be aligned on one common quarterly key?
4. What should happen to a quarter with missing inflation data?
5. Which storage format should be used for the integrated briefing dataset?

### Required Deliverables

- A DataFrame named `macro_data`.
- A missing-data audit.
- An SQLite table `macro_quarterly`.
- An Excel workbook for briefing.
- A short 100–150 word data-preparation note.

## 2.1 Read Inflation from CSV

### Tasks

1. Read `inflation.csv`.
2. Inspect shape and missing values.
3. Verify that `Quarter` is unique.

In [ ]:
# TODO 2.1
# inflation_2 = pd.read_csv(...)
# print(inflation_2.head())
# print(inflation_2.isna().sum())
# print("Quarter unique:", ...)

## 2.2 Read and Normalize Unemployment JSON

The JSON uses:

```text
period
└── labor_market
    └── unemployment_rate
```

### Tasks

1. Read the JSON file.
2. Normalize it to a DataFrame.
3. Rename fields to:
   - `Quarter`;
   - `Unemployment`.

In [ ]:
# TODO 2.2
# with open(SOURCE_DIR / "unemployment.json", encoding="utf-8") as f:
#     unemployment_raw = json.load(f)
#
# unemployment_2 = pd.json_normalize(...)
# unemployment_2 = unemployment_2.rename(columns={...})
#
# print(unemployment_2.head())

## 2.3 Extract GDP Growth from HTML

### Tasks

1. Use `pd.read_html()`.
2. Determine the number of tables.
3. Select the correct table.
4. Rename:
   - `Period` → `Quarter`;
   - `GDP Growth (%)` → `GDP_Growth`.

In [ ]:
# TODO 2.3
# tables_2 = pd.read_html(...)
# print("Number of tables:", ...)
#
# gdp_2 = ...
# gdp_2 = gdp_2.rename(columns={...})
# print(gdp_2.head())

## 2.4 Audit Coverage Before Integration

Create a set of quarters from each source.

### Questions

- Are all sets identical?
- Is any quarter missing entirely from one source?
- Is the missing inflation observation a **missing row** or a **missing value within an existing row**?

In [ ]:
# TODO 2.4
# q_inflation = set(...)
# q_unemployment = set(...)
# q_gdp = set(...)
#
# print("Inflation only:", ...)
# print("Unemployment only:", ...)
# print("GDP only:", ...)

## 2.5 Build the Integrated Macro Dataset

### Tasks

1. Merge the three sources on `Quarter`.
2. Use an appropriate join.
3. Name the result `macro_data`.
4. Inspect missing values after integration.
5. Do **not** silently replace missing inflation with 0.

In [ ]:
# TODO 2.5
# macro_data = (
#     inflation_2
#     .merge(...)
#     .merge(...)
# )
#
# print(macro_data)
# print(macro_data.isna().sum())

### Data Interpretation Question

Why would replacing missing inflation with `0` be analytically dangerous?

> ...

## 2.6 Store the Briefing Dataset

### Requirements

1. Save `macro_data` to SQLite table `macro_quarterly`.
2. Export to `economic_briefing.xlsx`.
3. Create a second Excel worksheet named `Missing_Data_Audit`.

In [ ]:
# TODO 2.6
# missing_audit_2 = ...
#
# conn = sqlite3.connect(OUTPUT_DIR / "economic_data.db")
# macro_data.to_sql(...)
# conn.close()
#
# with pd.ExcelWriter(
#     OUTPUT_DIR / "economic_briefing.xlsx",
#     engine="openpyxl"
# ) as writer:
#     ...

### Mini-Project 2 Commentary

Write **100–150 words** explaining:

- which source required the most transformation;
- whether all quarters aligned;
- what missing-data issue was detected;
- why preserving missingness may be better than replacing it automatically;
- why the integrated table is more useful than the original sources.

> ...

# MINI-PROJECT 3 — DATA SCIENCE
## Build and Validate a Modeling Dataset Before Machine Learning

### Scenario

A Data Science team is preparing a Customer Churn project.

The required information is distributed across:

- customer master → CSV;
- behavioral features → semicolon-delimited CSV;
- churn labels → SQLite.

Before modeling, the team needs a **single validated modeling dataset**.

### Why This Matters

A Machine Learning pipeline can fail silently if:

- IDs are inconsistent;
- duplicate customers exist;
- rows disappear during joins;
- target labels are missing;
- date fields remain strings;
- numerical fields are imported incorrectly.

### Modeling Dataset Requirements

The final dataset should contain:

- CustomerID;
- Age;
- City;
- JoinDate;
- Plan;
- Tenure;
- MonthlyCharge;
- SupportCalls;
- LastLoginDays;
- Churn.

### Required Deliverables

- A DataFrame named `modeling_data`.
- A data-quality report.
- Executable validation assertions.
- A modeling CSV file.
- A SQLite copy for reproducibility.

> This project stops before model training. The focus is **data readiness for Machine Learning**.

## 3.1 Read the Customer Master

### Tasks

1. Read `ds_customer_master.csv`.
2. Keep `CustomerID` as string.
3. Parse `JoinDate`.
4. Inspect missing values.
5. Check key uniqueness.

In [ ]:
# TODO 3.1
# customer_master_3 = pd.read_csv(
#     SOURCE_DIR / "ds_customer_master.csv",
#     dtype={...},
#     parse_dates=[...]
# )
#
# print(customer_master_3.info())
# print(customer_master_3.isna().sum())
# print("CustomerID unique:", ...)

## 3.2 Read the Behavioral Data

This file uses a **semicolon** delimiter.

### Tasks

1. Read `ds_behavior.csv`.
2. Use `sep=";"`.
3. Inspect data types.
4. Validate `CustomerID`.

In [ ]:
# TODO 3.2
# behavior_3 = pd.read_csv(
#     SOURCE_DIR / "ds_behavior.csv",
#     sep=...,
#     dtype={"CustomerID": str}
# )
#
# print(behavior_3.head())
# print(behavior_3.dtypes)

## 3.3 Read Churn Labels from SQLite

### Tasks

1. Connect to `ml_labels.db`.
2. Read table `churn_labels`.
3. Check the class distribution.
4. Confirm that each CustomerID has one label.

In [ ]:
# TODO 3.3
# conn = sqlite3.connect(SOURCE_DIR / "ml_labels.db")
# labels_3 = pd.read_sql_query(
#     "SELECT * FROM churn_labels",
#     conn
# )
# conn.close()
#
# print(labels_3["Churn"].value_counts())
# print("CustomerID unique:", ...)

## 3.4 Validate Source Coverage

Before merging, compare CustomerID sets.

### Questions

- Does every customer in the master have behavior data?
- Does every customer have a churn label?
- Are there any extra IDs in behavior or labels?

In [ ]:
# TODO 3.4
# master_ids = set(...)
# behavior_ids = set(...)
# label_ids = set(...)
#
# print("Missing behavior:", ...)
# print("Missing labels:", ...)
# print("Extra behavior IDs:", ...)
# print("Extra label IDs:", ...)

## 3.5 Build the Modeling Dataset

### Tasks

1. Merge customer master, behavior, and labels.
2. Use key validation.
3. Name the result `modeling_data`.
4. Check row count before and after.
5. Inspect missing values.

In [ ]:
# TODO 3.5
# modeling_data = (
#     customer_master_3
#     .merge(
#         behavior_3,
#         on="CustomerID",
#         how="left",
#         validate="one_to_one"
#     )
#     .merge(
#         labels_3,
#         on="CustomerID",
#         how="left",
#         validate="one_to_one"
#     )
# )
#
# print(modeling_data.shape)
# print(modeling_data.isna().sum())

## 3.6 Create a Data-Quality Report

Build a table with one row per column containing:

- column name;
- dtype;
- number of missing values;
- missing percentage;
- number of unique values.

In [ ]:
# TODO 3.6
# quality_report_3 = pd.DataFrame({
#     "Column": modeling_data.columns,
#     "Dtype": [...],
#     "Missing": [...],
#     "MissingPct": [...],
#     "Unique": [...]
# })
#
# quality_report_3

## 3.7 Add Executable Validation Checks

Your final modeling dataset should satisfy important assumptions.

Complete assertions such as:

```python
assert modeling_data["CustomerID"].is_unique
assert modeling_data["Churn"].notna().all()
assert set(modeling_data["Churn"].unique()).issubset({0, 1})
```

Add at least **three additional assertions**.

In [ ]:
# TODO 3.7
# assert ...
# assert ...
# assert ...
#
# print("All validation checks passed.")

## 3.8 Store the Modeling Dataset

### Requirements

1. Export `modeling_data.csv` using UTF-8.
2. Save `modeling_data` to SQLite table `modeling_data`.
3. Export `quality_report_3` to Excel.

Optional:

- export to Parquet if `pyarrow` or `fastparquet` is available.

In [ ]:
# TODO 3.8
# modeling_data.to_csv(...)
#
# conn = sqlite3.connect(OUTPUT_DIR / "modeling_data.db")
# modeling_data.to_sql(...)
# conn.close()
#
# quality_report_3.to_excel(...)

### Mini-Project 3 Conclusion

Complete the table:

| Data-quality finding | Evidence | Risk for modeling | Recommended action |
|---|---|---|---|
| ... | ... | ... | ... |
| ... | ... | ... | ... |
| ... | ... | ... | ... |

Then answer:

> Why should a Data Scientist validate source relationships before model training?

> ...

# MINI-PROJECT 4 — SUPPLY CHAIN ANALYTICS
## Build an Operational Data Mart for Inventory and Inbound Shipments

### Scenario

A supply chain team needs a daily operational dataset combining:

- inventory by warehouse → Excel;
- suppliers → Excel;
- inbound shipments → nested JSON;
- warehouse metadata → SQLite.

The team wants to identify:

- SKUs below reorder point;
- delayed shipments;
- suppliers associated with delays;
- warehouses receiving critical SKUs.

### Decision Questions

1. Which SKUs are already below reorder point?
2. Which shipments arrived late?
3. How many days late was each delayed shipment?
4. Which supplier and warehouse are associated with each shipment line?
5. Can the final dataset support both operational queries and Excel reporting?

### Required Deliverables

- A normalized shipment-line table.
- A DataFrame named `supply_chain_data`.
- A critical inventory table.
- A delayed shipment table.
- An SQLite operational data mart.
- A multi-sheet Excel report.

## 4.1 Read the Supply Chain Excel Workbook

### Tasks

1. Read all worksheets.
2. Create:
   - `inventory_4`;
   - `suppliers_4`.
3. Validate `SKU` and `SupplierID`.

In [ ]:
# TODO 4.1
# sc_book = pd.read_excel(
#     SOURCE_DIR / "supply_chain_master.xlsx",
#     sheet_name=None,
#     engine="openpyxl"
# )
#
# inventory_4 = ...
# suppliers_4 = ...
#
# print("SKU unique:", ...)
# print("SupplierID unique:", ...)

## 4.2 Normalize Shipment JSON

Each shipment contains a nested list of shipment lines.

### Tasks

Create one row per `ShipmentID × SKU` containing:

- ShipmentID;
- SupplierID;
- ExpectedDate;
- ActualDate;
- SKU;
- Qty.

Parse both date fields as datetime.

In [ ]:
# TODO 4.2
# with open(SOURCE_DIR / "shipments.json", encoding="utf-8") as f:
#     shipments_raw_4 = json.load(f)
#
# shipment_lines_4 = pd.json_normalize(
#     shipments_raw_4,
#     record_path=[...],
#     meta=[...]
# )
#
# shipment_lines_4["ExpectedDate"] = pd.to_datetime(...)
# shipment_lines_4["ActualDate"] = pd.to_datetime(...)
#
# print(shipment_lines_4)

## 4.3 Calculate Delivery Delay

Create:

\[
DelayDays = ActualDate - ExpectedDate
\]

expressed in days.

Then create:

```text
Delayed = DelayDays > 0
```

In [ ]:
# TODO 4.3
# shipment_lines_4["DelayDays"] = (
#     shipment_lines_4["ActualDate"]
#     - shipment_lines_4["ExpectedDate"]
# ).dt.days
#
# shipment_lines_4["Delayed"] = ...
#
# print(shipment_lines_4[["ShipmentID", "DelayDays", "Delayed"]])

## 4.4 Read Warehouse Metadata from SQLite

### Tasks

1. Connect to `operations.db`.
2. Read table `warehouses`.
3. Inspect warehouse names and capacities.

In [ ]:
# TODO 4.4
# conn = sqlite3.connect(...)
# warehouses_4 = pd.read_sql_query(
#     "SELECT * FROM warehouses",
#     conn
# )
# conn.close()
#
# print(warehouses_4)

## 4.5 Build the Supply Chain Data Mart

Create `supply_chain_data` by combining:

1. shipment lines;
2. suppliers;
3. inventory;
4. warehouse metadata.

Use `validate=` where appropriate.

In [ ]:
# TODO 4.5
# supply_chain_data = (
#     shipment_lines_4
#     .merge(
#         suppliers_4,
#         on="SupplierID",
#         how="left",
#         validate="many_to_one"
#     )
#     .merge(
#         inventory_4,
#         on="SKU",
#         how="left",
#         validate="many_to_one"
#     )
#     .merge(
#         warehouses_4,
#         on="Warehouse",
#         how="left",
#         validate="many_to_one"
#     )
# )
#
# print(supply_chain_data)
# print(supply_chain_data.isna().sum())

## 4.6 Create Operational Exception Tables

### Critical Inventory

A SKU is critical when:

```text
OnHand <= ReorderPoint
```

### Delayed Shipments

A shipment line is delayed when:

```text
DelayDays > 0
```

Create:

- `critical_inventory_4`;
- `delayed_shipments_4`.

In [ ]:
# TODO 4.6
# critical_inventory_4 = ...
# delayed_shipments_4 = ...
#
# print("Critical inventory:")
# print(critical_inventory_4)
#
# print("\nDelayed shipments:")
# print(delayed_shipments_4)

## 4.7 Use a Parameterized Operational Query

Store the integrated dataset in SQLite table `supply_chain_data`.

Then write a parameterized query to retrieve rows for a warehouse chosen by:

```python
warehouse_name = "Central"
```

Do not build the SQL query by directly concatenating the warehouse name into the SQL string.

In [ ]:
# TODO 4.7
# conn = sqlite3.connect(OUTPUT_DIR / "supply_chain_data.db")
# supply_chain_data.to_sql(
#     "supply_chain_data",
#     conn,
#     if_exists="replace",
#     index=False
# )
#
# warehouse_name = "Central"
#
# central_rows = pd.read_sql_query(
#     "SELECT * FROM supply_chain_data WHERE Warehouse = ?",
#     conn,
#     params=(warehouse_name,)
# )
#
# conn.close()
#
# print(central_rows)

## 4.8 Export the Operations Report

Create `supply_chain_report.xlsx` containing:

- `Integrated_Data`;
- `Critical_Inventory`;
- `Delayed_Shipments`.

### Final Questions

- Which data issues would make a daily supply chain report unreliable?
- Why is SQLite useful for recurring operational queries?
- Why is Excel still useful even if the analytical data are stored in SQL?

In [ ]:
# TODO 4.8
# with pd.ExcelWriter(
#     OUTPUT_DIR / "supply_chain_report.xlsx",
#     engine="openpyxl"
# ) as writer:
#     ...

### Mini-Project 4 Conclusion

Write **3–5 operational insights**:

1. ...
2. ...
3. ...

For each, distinguish between:

- what the integrated data directly show;
- what would require additional operational investigation.

# FINAL SECTION — Project Synthesis

After completing the four mini-projects, review the work as a **data engineering and analytical preparation process**, not as a checklist of file-reading functions.

A reliable analytical workflow is:

**Decision question → Data requirements → Source access → Inspection → Validation → Transformation → Integration → Storage → Reuse**

## 1. Techniques Used

Mark the techniques you completed:

- [ ] `pd.read_csv()`
- [ ] `sep=`
- [ ] `dtype=`
- [ ] `parse_dates=`
- [ ] `na_values=`
- [ ] `pd.read_excel()`
- [ ] `sheet_name=None`
- [ ] `pd.ExcelWriter()`
- [ ] `json.load()`
- [ ] `pd.json_normalize()`
- [ ] `pd.read_html()`
- [ ] `sqlite3.connect()`
- [ ] `pd.read_sql_query()`
- [ ] parameterized SQL
- [ ] `DataFrame.to_sql()`
- [ ] merge-key validation
- [ ] `validate=` in `merge()`
- [ ] post-merge missing-value audit
- [ ] executable assertions
- [ ] multi-source analytical dataset
- [ ] multi-sheet Excel export

### More Important Self-Assessment

For every data source, ask:

> **Why is this source needed, and what role does it play in the final analytical dataset?**

For every merge, ask:

> **What relationship between the two tables am I assuming?**

For every output file, ask:

> **Who will use this output, and why is this storage format appropriate?**

## 2. Synthesis Questions

### Question 1

What is the difference between **data access** and **data integration**?

> ...

### Question 2

Why should identifier fields often be stored as strings?

> ...

### Question 3

Why is checking only `df.isna().sum()` not enough to validate an integrated dataset?

> ...

### Question 4

When should an analyst prefer SQL storage over CSV?

> ...

### Question 5

Why should a missing value not automatically be replaced with zero?

> ...

### Question 6

Which mini-project required the most important data validation step? Explain.

> ...

# Suggested Grading Rubric — 100 Points

| Component | Points |
|---|---:|
| Mini-Project 1 — Retail Data Integration | 25 |
| Mini-Project 2 — Economic Data Harmonization | 20 |
| Mini-Project 3 — Modeling Dataset Preparation | 25 |
| Mini-Project 4 — Supply Chain Data Mart | 20 |
| Data validation and reproducibility | 5 |
| Interpretation and storage rationale | 5 |
| **Total** | **100** |

## Quality Criteria

A strong submission does not only contain code that runs.

It should demonstrate that the learner can explain:

- what each source represents;
- why a particular import option is used;
- how keys are validated;
- why a join type is appropriate;
- what data-quality issues were detected;
- why a storage format was selected;
- whether the final analytical dataset can be reproduced from the raw sources.

# Optional Extension Challenges

Choose **one** extension if you finish early.

### Challenge A — Parquet Output

Export one large analytical DataFrame to Parquet and compare:

- file size;
- read time;
- preservation of data types;

with CSV.

### Challenge B — MongoDB-Style Storage

Convert one analytical dataset into a list of nested dictionaries suitable for document storage.

Optional: if a local MongoDB server is available, insert the documents with PyMongo.

### Challenge C — PDF Table Extraction

Create or obtain a PDF containing a simple table and attempt to extract it with `pdfplumber`.

Document:

- what was extracted correctly;
- what required cleaning;
- why PDF is less reliable than CSV/Excel for structured analytical data.

### Challenge D — Data Validation Function

Write a reusable function:

```python
validate_dataset(df, key_columns, required_columns)
```

that reports:

- duplicate keys;
- missing required fields;
- data types;
- row count;
- unique counts.

# Conclusion

The central idea of this project is:

> **Data are not ready for analysis simply because Python can read the file.**

A trustworthy analytical dataset requires:

```text
Source Identification
        ↓
Correct Import
        ↓
Structural Inspection
        ↓
Key Validation
        ↓
Transformation
        ↓
Integration
        ↓
Post-Merge Validation
        ↓
Appropriate Storage
        ↓
Reproducible Reuse
```

The next step is to replace the simulated project sources with real organizational or public datasets while preserving the same validation and integration discipline.

In [ ]:
print("Project notebook loaded.")
print("Source directory:", SOURCE_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())
print("Complete the TODO cells and save your final notebook.")